In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Convert to Tensors: Convert the numpy arrays (X_train, X_test, y_train, y_test) into PyTorch tensors.
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)


In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)


In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
# 4. Print shape of one batch.
images_batch, ages_batch = next(iter(train_loader))
print("Train batch images shape:", images_batch.shape)
print("Train batch ages shape:", ages_batch.shape)



In [ ]:
# 5. Display sample images

images_batch_show = images_batch[:6]
ages_batch_show = ages_batch[:6]

fig, axes = plt.subplots(2, 3, figsize=(10, 7))
axes = axes.flatten()
for i in range(len(images_batch_show)):
    img = images_batch_show[i].permute(1, 2, 0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(f"Actual age: {ages_batch_show[i].item():.0f}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Task 1: Write your model class here:
class AgeRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3 * 36 * 36, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x


In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    total = 0
    for images_batch, ages_batch in loader:
        images_batch = images_batch.to(device)
        ages_batch = ages_batch.to(device).float()
        optimizer.zero_grad()
        outputs = model(images_batch).squeeze(1)
        loss = criterion(outputs, ages_batch)
        loss.backward()
        optimizer.step()
        batch_size = images_batch.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size
    epoch_loss = running_loss / total
    return epoch_loss



In [ ]:
# Task 3: Write your validation loop here:
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    total = 0
    with torch.no_grad():
        for images_batch, ages_batch in loader:
            images_batch = images_batch.to(device)
            ages_batch = ages_batch.to(device).float()
            outputs = model(images_batch).squeeze(1)
            loss = criterion(outputs, ages_batch)
            batch_size = images_batch.size(0)
            running_loss += loss.item() * batch_size
            total += batch_size
    epoch_loss = running_loss / total
    return epoch_loss


In [ ]:
# Task 4: Define device, model, loss, optimizer:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device used for training:", device)

model = AgeRegressor().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 40
train_losses = []
val_losses = []

print("Starting training...")
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = evaluate(model, test_loader, criterion, device)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch + 1}/{num_epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")
print("Training finished.")


In [ ]:

# Plot the training and validation loss over epochs.
epochs_range = range(1, num_epochs + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs_range, train_losses, label="Train Loss")
plt.plot(epochs_range, val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss over Epochs")
plt.legend()
plt.grid(True)
plt.show()




In [ ]:
# Task 1: Write your code here:

model.eval()
images_batch, ages_batch = next(iter(test_loader))
images_batch_device = images_batch.to(device)
ages_batch_device = ages_batch.to(device)

with torch.no_grad():
    preds = model(images_batch_device).squeeze(1)

images_batch = images_batch.cpu()
ages_batch = ages_batch.cpu()
preds = preds.cpu()



In [ ]:
# Task 2 (Bonus): Write your code here:
num_show = 6
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
axes = axes.flatten()
for i in range(num_show):
    img = images_batch[i].permute(1, 2, 0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(f"Actual: {ages_batch[i].item():.0f} - Predicted: {preds[i].item():.1f}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()
